In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate sacrebleu
!pip install -q tokenizers pandas scikit-learn

import os
import gc
import csv
import glob
import shutil
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    NllbTokenizerFast,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed
)
import evaluate

# =========================================================
# 1. GLOBAL CONFIGURATION & DATA PREPARATION
# =========================================================
MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
CSV_DATA_PATH = "dataset_terjemahan_no_word_nosentence.csv"

# Experimental Matrix: 3 conditions x 10 seeds = 30 total runs
EXPERIMENT_MODES = ["baseline", "mean_only", "norm_mean"]
#SEEDS = [42, 123, 456, 789, 1024]
SEEDS = [42, 123, 456, 789, 1024, 2024, 3141, 4242, 8080, 9999]

# Define your target glottal syllables
#CUSTOM_GLOTTAL_TOKENS = ["'u", "'a", "'e", "'i", "'o","'at","'li","'an","'we","'on","'it","'in","'en","'wo","'ak","'ro","'lo","'er","'eng","'un"]
CUSTOM_GLOTTAL_TOKENS = ["'u", "'a", "'e", "'i", "'o"]
SPECIAL_LANG_TOKEN = "sik_Latn"

# Load & Clean Dataset
df = pd.read_csv(CSV_DATA_PATH, delimiter=';', dtype=str)
target_cols = ['bahasa_sikka', 'bahasa_indonesia']
df = df.dropna(subset=target_cols)
df['bahasa_sikka'] = df['bahasa_sikka'].astype(str).str.strip()
df['bahasa_indonesia'] = df['bahasa_indonesia'].astype(str).str.strip()
df = df[(df['bahasa_sikka'] != "") & (df['bahasa_indonesia'] != "")]

raw_dataset = Dataset.from_pandas(df[['bahasa_sikka', 'bahasa_indonesia']])
# Seed 42 locked for split to ensure identical evaluation sets across all runs
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

# =========================================================
# 2. VECTOR EXTRACTION (Calculated naive mean and norm mean from base weights)
# =========================================================
def precalculate_vectors(tokens, model_name):
    """Computes both unscaled mean and GATI norm-scaled vectors."""
    base_tok = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    embed_weights = base_model.get_input_embeddings().weight.detach()

    payload = {}
    for token in tokens:
        sub_ids = base_tok(token, add_special_tokens=False)["input_ids"]
        sub_tensor = torch.tensor(sub_ids)
        sub_vecs = embed_weights[sub_tensor]

        # Phase 1A: Raw Mean
        raw_mean = torch.mean(sub_vecs, dim=0)

        # Phase 1B: Norm Target
        target_norm = torch.mean(torch.norm(sub_vecs, dim=1))

        # Phase 1C: Norm-Scaled (GATI)
        norm_scaled = (raw_mean / torch.norm(raw_mean)) * target_norm

        payload[token] = {
            "shattered_tokens": base_tok.convert_ids_to_tokens(sub_ids),
            "mean_only": raw_mean,
            "norm_mean": norm_scaled
        }

    del base_model, base_tok, embed_weights
    torch.cuda.empty_cache()
    return payload

print("Pre-calculating representation vectors...")
INJECTION_PAYLOAD = precalculate_vectors(CUSTOM_GLOTTAL_TOKENS, MODEL_CHECKPOINT)

# =========================================================
# 3. METRIC & CALLBACK DEFINITIONS
# =========================================================
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

class CSVMetricsLogger(TrainerCallback):
    def __init__(self, filepath):
        self.filepath = filepath
        if not os.path.exists(self.filepath):
            with open(self.filepath, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['epoch', 'step', 'train_loss', 'eval_loss', 'eval_bleu', 'eval_chrf'])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and 'eval_loss' in logs:
            train_loss = next((l.get("loss") for l in reversed(state.log_history) if "loss" in l), "")
            with open(self.filepath, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    round(state.epoch, 2) if state.epoch else '',
                    state.global_step,
                    train_loss,
                    logs.get('eval_loss', ''),
                    logs.get('eval_bleu', ''),
                    logs.get('eval_chrf', '')
                ])

# =========================================================
# 4. SINGLE RUN EXECUTION UNIT
# =========================================================
def execute_single_run(mode: str, seed: int):
    output_dir = f"./nllb-indo-sikka-{mode}-seed-{seed}"
    csv_filepath = f"./metrics_{mode}_seed_{seed}.csv"

    # Resumption guard: skip if already executed
    if os.path.exists(csv_filepath) and os.path.getsize(csv_filepath) > 120:
        print(f"Skipping {mode} (Seed {seed}): Log exists and is non-empty.")
        return

    print(f"\n{'='*70}\nSTARTING: Mode = {mode.upper()} | Seed = {seed}\n{'='*70}")
    os.makedirs(output_dir, exist_ok=True)
    set_seed(seed)

    # 1. Tokenizer Setup
    tokenizer = NllbTokenizerFast.from_pretrained(MODEL_CHECKPOINT)
    tokenizer.add_special_tokens({'additional_special_tokens': [SPECIAL_LANG_TOKEN]})
    tokenizer.add_tokens(CUSTOM_GLOTTAL_TOKENS)

    temp_tok_path = f"./tok_temp_{mode}_{seed}"
    tokenizer.save_pretrained(temp_tok_path)
    tokenizer = AutoTokenizer.from_pretrained(temp_tok_path, use_fast=True)
    tokenizer.src_lang = "ind_Latn"
    tokenizer.tgt_lang = SPECIAL_LANG_TOKEN

    # 2. Model & Embedding Setup
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
    model.resize_token_embeddings(len(tokenizer))

    sik_id = tokenizer.convert_tokens_to_ids(SPECIAL_LANG_TOKEN)
    ind_id = tokenizer.convert_tokens_to_ids("ind_Latn")

    in_embeds = model.get_input_embeddings().weight
    out_embeds = model.get_output_embeddings().weight

    with torch.no_grad():
        # Source language alignment transfer
        in_embeds[sik_id] = in_embeds[ind_id]
        out_embeds[sik_id] = out_embeds[ind_id]

        # Injection branches
        if mode in ["mean_only", "norm_mean"]:
            for token, vectors in INJECTION_PAYLOAD.items():
                t_id = tokenizer.convert_tokens_to_ids(token)
                target_vec = vectors[mode].to(model.device)
                in_embeds[t_id] = target_vec
                out_embeds[t_id] = target_vec
        # "baseline" leaves newly resized embeddings at PyTorch default initialization

    # 3. Tokenize Dataset
    def preprocess_fn(examples):
        inputs = [str(s).strip() if s else " " for s in examples["bahasa_indonesia"]]
        targets = [str(s).strip() if s else " " for s in examples["bahasa_sikka"]]
        model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)

        labels = model_inputs["labels"]
        model_inputs["labels"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in labels
        ]
        return model_inputs

    tokenized_train = split_dataset["train"].map(preprocess_fn, batched=True, remove_columns=split_dataset["train"].column_names)
    tokenized_test = split_dataset["test"].map(preprocess_fn, batched=True, remove_columns=split_dataset["test"].column_names)

    # 4. Evaluation Function
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        dec_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
        dec_labels = [[l.strip()] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
        return {
            "bleu": bleu_metric.compute(predictions=dec_preds, references=dec_labels)["score"],
            "chrf": chrf_metric.compute(predictions=dec_preds, references=dec_labels)["score"]
        }

    # 5. Trainer Arguments & Setup
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        save_total_limit=1,
        num_train_epochs=50,
        predict_with_generate=True,
        fp16=True,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="chrf",
        greater_is_better=True,
        seed=seed,
        data_seed=42
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[
            #EarlyStoppingCallback(early_stopping_patience=30),
            CSVMetricsLogger(csv_filepath)
        ]
    )

    # 6. Execute Training
    trainer.train()

    # 7. Teardown & VRAM Recovery
    del trainer, model, tokenizer, in_embeds, out_embeds
    if os.path.exists(temp_tok_path):
        shutil.rmtree(temp_tok_path)
    # Remove checkpoints to preserve storage during multi-seed run
    shutil.rmtree(output_dir, ignore_errors=True)

    gc.collect()
    torch.cuda.empty_cache()
    print(f"Completed and logged: {csv_filepath}")

# =========================================================
# 5. EXECUTE AUTOMATED ABLATION PIPELINE
# =========================================================
for mode in EXPERIMENT_MODES:
    for seed in SEEDS:
        execute_single_run(mode, seed)

print("\nAll experimental runs have completed successfully.")